![](https://live.staticflickr.com/65535/54531085276_a4c5d63f4f_b.jpg)

*Images were generated using the Flux-dev model and then edited by the author of the task.*

# Introduction
The topic of this task is the issue of image enhancement, which is a collective term covering actions such as:

- image super-resolution,
- image denoising,
- image deblurring,
- low-light image enhancement,
- **image colorization**,
- and many others.

The goal of these techniques is to obtain a high-quality image based on a low-quality photo. This process often requires supplementing missing information in the image based on its context, therefore deep learning methods are commonly used.

Thanks to such techniques, we can, in a sense, 'look' into the past from a new perspective. An example is the photo of Warsaw from the beginning of the interwar period below – on the left in the original, and on the right in the colorized version by Mariusz Zając.

![](https://live.staticflickr.com/65535/54531426695_24b5710613_b.jpg)

In this task, we will focus on colorizing black-and-white photographs depicting human faces.

### Image Colorization
A color image is described by three RGB channels (red, green, blue). It can be easily converted into a monochromatic image, described by a single channel. However, when converting to grayscale, the simple arithmetic mean of RGB values is not used, because it does not correspond to how the human eye perceives the brightness of individual colors (e.g., blue light is perceived as darker than green).
For this reason, pixels in a grayscale image are calculated according to the formula:

$$ x^{(gray)} = 0.299 \cdot x^{(red)} + 0.587 \cdot x^{(green)} + 0.114 \cdot x^{(blue)}. $$

It is worth noting that the conversion from RGB to grayscale is unambiguous and easy to perform. The reverse operation – recovering color information – is not unambiguous, because many different combinations of RGB values can lead to the same grayscale value. This means that image colorization is an ill-posed problem, which implies that for a single black-and-white photo, many correct color versions can be generated.

For example, a black-and-white photo of a car is difficult to colorize unambiguously – the vehicle could have been almost any color. On the other hand, models trained on appropriate data can use statistical regularities – such as grass usually being green, the sky blue, and a tiger orange-black – to generate the most likely color versions.

Formally, the colorization task is defined as learning a model $\mathcal{M}_\theta$ with parameters $\theta$, which takes a grayscale image $x^{(gray)}$ as input and generates its color version $\hat{x}^{(rgb)}$:

$$ \hat{x}^{(rgb)} = \mathcal{M}_\theta(x^{(gray)}). $$

We want the model's predictions $\hat{x}^{(rgb)}$ to be as similar as possible to the original color images $x^{(rgb)}$. To this end, we aim to find parameter values $\theta$ that minimize a certain measure of difference (e.g., mean squared error) between predictions and real images:

$$ \theta^* = \arg \min_{\theta} \sum_{i = 1}^N d(x^{(rgb)}_i, \mathcal{M}_\theta(x^{(gray)}_i)), $$

where $N$ denotes the number of examples in the training set.

### Generative Adversarial Networks
Generative Adversarial Networks (GANs) were one of the first approaches addressing the problem of data generation using deep learning techniques. Because the generation process does not require any input to the network, it is not straightforward to propose a loss function, since we don't know what output to expect from the network. For example, the network might generate a plausible face image, but when compared to another random face photo, the penalty would be very large, despite the correctness of the network's output.

For this reason, the GAN architecture consists of two networks: a generator and a discriminator.

- The role of the generator is to generate new, random images.
- The role of the discriminator is to predict whether the input image comes from the dataset or was generated by the generator.

The generator tries to fool the discriminator (maximizes the discriminator's classification error), while the discriminator tries to guess whether the image is fake or not (minimizes its own classification error). Both networks are trained alternately until convergence is achieved.

In this task, we will not be training a GAN; instead, we will have a pre-trained generator model that can generate a random face image of resolution $256 \times 256$ based on noise. The architecture of this network is called StyleGAN and is presented in the diagram below. It is divided into two modules: a mapping network $f$ and a synthesis network $g$.

Network $f$ is an MLP architecture transforming a random vector $z$, having $512$ elements with values from the normal distribution $\mathcal{N}(0, \mathbb{I})$, into a $512$-dimensional vector $w$, which is used to condition network $g$. Vector $w$ can be interpreted as a latent representation of the generated image.

Network $g$ is a convolutional network with a constant $4 \times 4$ input, which progressively increases its resolution to $256 \times 256$. In each block, network $g$ is conditioned by vector $w$, ensuring the diversity of generated images. Additionally, some noise is added to the network, further increasing diversity.

![](https://live.staticflickr.com/65535/54531085226_a4fe9a0c2e_z.jpg)


# Task
In this task, having access to a pre-trained `generator` capable of generating face images of resolution $256 \times 256$, you need to propose a method for colorizing black-and-white face images of the same resolution. The goal is to extract the knowledge embedded in the generator's weights and use it for a completely different task, previously unknown to the generator.

### Data
In this task, there are no available training data, only validation data for the initial evaluation of your proposed approach. The validation data contains 500 paired black-and-white images with their color counterparts.

### Evaluation Criterion
To evaluate the quality of your solution, a metric consisting of two sub-metrics will be used: *PSNR* and *LPIPS*.

PSNR is inversely proportional to the mean squared error between the generated sample $\hat{x}^{(rgb)}$ and the real color image $x^{(rgb)}$. It is expressed by the formula:
$$ PSNR = 10 \cdot \log_{10}\left(\frac{\text{range}^2}{MSE(\hat{x}^{(rgb)}, x^{(rgb)})}\right), $$
where $\text{range}$ is the range of possible values that the images $\hat{x}^{(rgb)}$ and $x^{(rgb)}$ can take. We want to maximize PSNR.
The score range for this metric is $(22.0, 26.0)$ - points are awarded proportionally.

LPIPS is the average $L1$ distance between feature maps from selected layers of the `AlexNet` network, obtained from the inputs $\hat{x}^{(rgb)}$ and $x^{(rgb)}$. We want to minimize LPIPS.
The score range for this metric is $(0.15, 0.11)$ - points are awarded proportionally.

The final point value is the weighted average of points obtained from the PSNR and LPIPS metrics with weights $0.25$ (PSNR) and $0.75$ (LPIPS).

### Limitations

- Your solution will be tested on the Competition Platform without internet access and in a GPU environment.
- The evaluation of your final solution on the Competition Platform cannot take longer than 5 minutes with GPU.
- List of allowed libraries: torch, numpy, torchvision, pillow.

### Submission Files
Only this notebook, supplemented with your solution (see the `YourModel` class), should be submitted.

### Evaluation
During the check, the `FINAL_EVALUATION_MODE` flag will be set to `True`.

You can get between 0 and 100 points for this task. The number of points you earn will be calculated on the (secret) test set on the Competition Platform based on the above-mentioned formula, rounded to an integer. If your solution does not meet the above criteria or does not execute correctly, you will receive 0 points for the task.

# Starter Code

In [1]:
######################### DO NOT CHANGE THIS CELL ##########################

FINAL_EVALUATION_MODE = False

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

import os
import torch
import tarfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torchvision.transforms as T

from PIL import Image
from io import BytesIO
from torch.utils.data import Dataset, DataLoader
from torchmetrics.image import PeakSignalNoiseRatio as PSNR
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity as LPIPS

from stylegan import load_generator
RANDOM_SEED = 1

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False

The following cell contains evaluation and visualization functions for your solutions, as well as the validation dataset.

In [3]:
######################### DO NOT CHANGE THIS CELL ##########################

import math

def round_half_up(number: float) -> int:
    return int(math.floor(number + 0.5))

def plot_batch(batch):
    """ Function to display a batch of images (generated or real). Shows a maximum of 8 images """
    to_show = min(len(batch), 8)

    batch = batch.to('cpu')
    batch = batch * 0.5 + 0.5
    batch = batch.clamp(0, 1)
    batch = batch.permute(0, 2, 3, 1).numpy()

    fig, axes = plt.subplots(1, to_show, figsize=(to_show * 3, 3))

    if to_show == 1:
        axes = [axes]

    for ax, img in zip(axes, batch[:to_show]):
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


class TarImageDataset(Dataset):
    """ Dataset class, used for validating your method """
    def __init__(self, tar_path):
        self.tar_path = tar_path
        self.tar = tarfile.open(tar_path, 'r')
        self.gt_paths = sorted([m.name for m in self.tar.getmembers() if m.name.startswith('data/GT/') and m.name.endswith('.jpeg')])
        self.gray_paths = [p.replace('GT', 'GRAY') for p in self.gt_paths]

        self.transform = T.Compose([T.ToTensor(), T.Normalize(mean=[0.5], std=[0.5])])

    def __len__(self):
        return len(self.gt_paths)

    def __getitem__(self, idx):
        """ returns a monochromatic image and a color image """
        gt_member = self.tar.getmember(self.gt_paths[idx])
        gray_member = self.tar.getmember(self.gray_paths[idx])

        gt_image = Image.open(BytesIO(self.tar.extractfile(gt_member).read())).convert('RGB')
        gray_image = Image.open(BytesIO(self.tar.extractfile(gray_member).read())).convert('L')

        return (
            self.transform(gray_image),
            self.transform(gt_image)
        )

    def __del__(self):
        """ Closes the file when an object of this class is deleted """
        if hasattr(self, 'tar') and self.tar:
            self.tar.close()


def validate_solution(your_model, device, split="val"):
    dataset = TarImageDataset(f"./data/{split}.data")
    dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

    lpips_metric = LPIPS(net_type='alex').to(device)
    psnr_metric = PSNR(data_range=2.0).to(device)

    your_model.eval()

    for x_gray, x_gt in dataloader:
        x_gray = x_gray.to(device)
        x_gt = x_gt.to(device)

        x_pred = your_model.predict(x_gray).clamp_(-1, 1)

        psnr_metric.update(x_pred, x_gt)
        lpips_metric.update(x_pred, x_gt)

    avg_psnr = psnr_metric.compute()
    avg_lpips = lpips_metric.compute()

    print("PSNR: ", avg_psnr)
    print("LPIPS:", avg_lpips)

    PSNR_MIN, PSNR_MAX = 22, 26
    LPIPS_MIN, LPIPS_MAX = 0.11, 0.15

    psnr_points = ((torch.clamp(avg_psnr, PSNR_MIN, PSNR_MAX) - PSNR_MIN) / (PSNR_MAX - PSNR_MIN)).item()
    lpips_points = ((LPIPS_MAX - torch.clamp(avg_lpips, LPIPS_MIN, LPIPS_MAX)) / (LPIPS_MAX - LPIPS_MIN)).item()
    total_points = psnr_points * 0.25 + lpips_points * 0.75

    psnr_points = round_half_up(psnr_points * 100)
    lpips_points = round_half_up(lpips_points * 100)
    total_points = round_half_up(total_points * 100)

    return psnr_points, lpips_points, total_points


def show_image_grid(inputs, predictions, targets):
    """
    function to visualize inputs, predictions and target images
    """
    def tensor_to_numpy(img):
        img = (img.clamp(-1, 1) + 1) / 2
        img = img.cpu().numpy()
        if img.shape[0] == 1:
            return img.squeeze(0)
        return np.transpose(img, (1, 2, 0))

    titles = ["Inputs", "Model predictions", "Target images"]
    images = [inputs, predictions, targets]

    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    for row in range(3):
        for col in range(4):
            img = tensor_to_numpy(images[row][col])
            cmap = 'gray' if images[row].shape[1] == 1 else None
            axes[row, col].imshow(img, cmap=cmap)
            axes[row, col].axis('off')
        axes[row, 0].text(-0.2, 0.5, titles[row], va='center', ha='right',
                          fontsize=14, transform=axes[row, 0].transAxes)

    plt.suptitle("Example images from the validation set")
    plt.tight_layout()
    plt.show()


def show_score_bars(psnr_score, lpips_score, task_score):
    """
    Function to visualize the obtained points in the form of bar charts
    """
    labels = ["points for PSNR", "points for LPIPS", "points for task"][::-1]
    values = [psnr_score, lpips_score, task_score][::-1]

    plt.style.use('ggplot')

    fig, ax = plt.subplots(figsize=(16, 4))
    y = np.arange(len(labels))
    norm = mcolors.Normalize(vmin=0, vmax=100)
    cmap = plt.get_cmap('RdYlGn')
    colors = [cmap(norm(v)) for v in values]

    ax.barh(y, values, color=colors)
    ax.set_xlim(0, 100)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Points")
    ax.set_title("Model scores")
    for i, v in enumerate(values):
        ax.text(v + 1, i, f"{v:.1f}", va='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    # Reset to default style
    plt.style.use('default')


def benchmark_solution(your_model, device, split="val"):
    """
    Function that validates the model on the validation data and visualizes the obtained results
    """
    dataset = TarImageDataset(f"./data/{split}.data")
    print(len(dataset))
    dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

    x_gray, x_gt = next(iter(dataloader)) 
    x_pred = your_model.predict(x_gray.to(device)).cpu().clamp_(-1, 1)

    psnr_points, lpips_points, total_points = validate_solution(your_model, device, split)
    
    show_image_grid(x_gray, x_pred, x_gt)
    show_score_bars(psnr_points, lpips_points, total_points)



In [4]:
######################### DO NOT CHANGE THIS CELL ##########################

latent_dim = 512   # size of the latent representation of the StyleGAN model
size = 256         # resolution of images generated by the StyleGAN model
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if device == 'cpu':
    print('Warning: no GPU on the machine!')

# Loading the StyleGAN generator.
# Its implementation is in the stylegan.py file.
# Detailed analysis of the model code is not prohibited, but is not suggested
generator = load_generator()
generator.to(device)

print('Model loaded successfully')

Model loaded successfully


The following cell contains code for generating faces using the StyleGAN model. Generation is presented in two versions. The first version is more detailed and consists of the following steps:

- first we generate $z$ from a normal distribution
- then we use the `get_latent` method to generate $w$ vectors using the $f(z)$ model
- finally, we call the `style_to_image` method, which uses the $g$ network to generate face images conditioned on the $w$ vector.

Alternatively, we can generate images using the `forward` method, which combines steps 2 and 3.

In [5]:
# ######################### DO NOT CHANGE THIS CELL ##########################

# if not FINAL_EVALUATION_MODE:
#     with torch.no_grad():
#         z = torch.randn(8, latent_dim, device=device) 
#         w = generator.get_latent(z)
#         sample = generator.style_to_image(w)
#         plot_batch(sample)

#         z = torch.randn(8, latent_dim, device=device)
#         sample = generator.forward(z)
#         plot_batch(sample)


## Your Solution
Implement your solution in the cell below. Ensure that the `fit` method prepares the colorizing model, while the `predict` method uses the model to colorize the input `x_gray`.

In [6]:
from torch import nn

class YourModel(torch.nn.Module):
    def __init__(self, generator):
        super().__init__()
        self.generator = generator
        self.generator.eval()
        for p in self.generator.parameters():
            p.requires_grad_(False)

    def fit(self):
        self.generator.eval()
        with torch.no_grad():
            z_samples = torch.randn(2000, latent_dim, device=device)
            w_samples = self.generator.get_latent(z_samples)
            self.w_mean = w_samples.mean(0)

    def predict(self, x_gray):
        batch_size = x_gray.shape[0]
        EPOCHS = 2
        gray_filter = torch.tensor([0.299, 0.587, 0.114]).reshape(1,3,1,1).to(device)

        w_init = self.w_mean.unsqueeze(0).expand(batch_size, -1).clone()
        w = nn.Parameter(w_init.detach().requires_grad_(True))
        optimizer = torch.optim.Adam([w], lr=1e-1)

        lam = 0.002
        w_anchor = w_init.detach()
        criterion = nn.MSELoss()
        # lpips_metric = LPIPS(net_type='alex').to(device)

        for epoch in range(EPOCHS):
            optimizer.zero_grad()

            with torch.enable_grad():
                pred_color = self.generator.style_to_image(w)
            pred = (pred_color * gray_filter).sum(dim=1, keepdim=True)

            # lpips_loss = lpips_metric(pred, x_gray)
            mse_loss = criterion(pred, x_gray)
            loss_reg = lam * criterion(w, w_anchor)
            loss = mse_loss + loss_reg
            loss.backward()
            optimizer.step()

            if epoch % 10 == 9:
                print(f'Epochs {epoch+1} | Regularization Loss: {loss_reg.item()} | MSE Loss: {mse_loss.item()}')


        with torch.no_grad():
            res = self.generator.style_to_image(w)

        return res.clamp(-1,1)

In [7]:
######################### DO NOT CHANGE THIS CELL ##########################

your_model = YourModel(generator)
your_model.fit()

if not FINAL_EVALUATION_MODE:
    benchmark_solution(your_model, device)

250


KeyboardInterrupt: 